<a href="https://colab.research.google.com/github/zombie9088/Pytorch_learning/blob/main/Training_first_NN_gpu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt


In [ ]:
torch.manual_seed(42)

In [ ]:
device= torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/fmnist/fashion-mnist_train.csv")
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,6,0,0,0,0,0,0,0,5,0,...,0,0,0,30,43,0,0,0,0,0
3,0,0,0,0,1,2,0,0,0,0,...,3,0,0,0,0,1,0,0,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
X= df.iloc[:,1:].values
y= df.iloc[:, 0].values

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [ ]:
X_train = X_train/255
X_test= X_test/255

In [ ]:
#custom datset class

class CustomDataset(Dataset):
  def __init__(self,features,labels):
    self.features=torch.tensor(features,dtype=torch.float32)
    self.labels= torch.tensor(labels,dtype=torch.long)

  def __len__(self):
    return len(self.features)

  def __getitem__(self, index):
    return self.features[index], self.labels[index]


In [ ]:
#creating train dataset object

train_dataset = CustomDataset(X_train,y_train)


In [ ]:
#create test dataset object
test_dataset= CustomDataset(X_test,y_test)

In [ ]:
#create train and test dataloader

train_loader= DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader= DataLoader(test_dataset,batch_size=32,shuffle=False)

In [ ]:
#define nn class

class MyNN(nn.Module):
  def __init__(self,num_features):
    super().__init__()

    self.model= nn.Sequential(
        nn.Linear(num_features,128),
        nn.ReLU(),
        nn.Linear(128,64),
        nn.ReLU(),
        nn.Linear(64,10)
    )

  def forward(self,x):
    return self.model(x)


In [ ]:
#epochs and learning rate

epochs= 100
learning_rate = 0.1

In [ ]:
#intiate the model
model = MyNN(X_train.shape[1])
model= model.to(device)

#loss function

criterion = nn.CrossEntropyLoss()

#optimiser

optimizer= optim.SGD(model.parameters(),lr=learning_rate)

In [ ]:
#training loop

for epoch in range(epochs):

  total_epoch_loss=0
  for batch_features, batch_labels in train_loader:
    batch_features= batch_features.to(device)
    batch_labels= batch_labels.to(device)

    #forward pass
    outputs= model(batch_features)

    #loss
    loss= criterion(outputs,batch_labels)

    optimizer.zero_grad()

    #backprop
    loss.backward()


    #update gradient
    optimizer.step()
    total_epoch_loss= total_epoch_loss+loss.item()

  print(f"Epoch {epoch+1}/{epochs}, Loss: {total_epoch_loss/len(train_loader)}")


Epoch 1/100, Loss: 0.648179722537597
Epoch 2/100, Loss: 0.4376127086083094
Epoch 3/100, Loss: 0.3899180464297533
Epoch 4/100, Loss: 0.35898561184853317
Epoch 5/100, Loss: 0.34057426101962723
Epoch 6/100, Loss: 0.32460947323093814
Epoch 7/100, Loss: 0.30800089277823767
Epoch 8/100, Loss: 0.2985224675362309
Epoch 9/100, Loss: 0.2861548168013493
Epoch 10/100, Loss: 0.2764378187954426
Epoch 11/100, Loss: 0.26789126490056514
Epoch 12/100, Loss: 0.2605256137425701
Epoch 13/100, Loss: 0.25143250030527514
Epoch 14/100, Loss: 0.2450787955534955
Epoch 15/100, Loss: 0.23941551821182172
Epoch 16/100, Loss: 0.23350247728079557
Epoch 17/100, Loss: 0.22620921324566007
Epoch 18/100, Loss: 0.22178158815701804
Epoch 19/100, Loss: 0.21645093370849888
Epoch 20/100, Loss: 0.21166922993088763
Epoch 21/100, Loss: 0.207785743618384
Epoch 22/100, Loss: 0.20269458599823217
Epoch 23/100, Loss: 0.19785277682418626
Epoch 24/100, Loss: 0.19349404489093772
Epoch 25/100, Loss: 0.19209285398448506
Epoch 26/100, Loss: 

In [ ]:
#set model to eval mode

model.eval()


MyNN(
  (model): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [ ]:
#evaluation code

total=0
correct =0

with torch.no_grad():
  for batch_features, batch_labels in test_loader:
    batch_features= batch_features.to(device)
    batch_labels= batch_labels.to(device)

    outputs= model(batch_features)

    _,predicted = torch.max(outputs,1)
    total+= batch_labels.shape[0]
    correct+= (predicted==batch_labels).sum().item()

print(correct/total)

0.8885
